# GeoNUSAF Part 2 / Stage 4 — SegNeXt-T, real vs real+synthetic (R0 / R1)

**Block fold 1. One arm per session. Change `ARM` in section 2 and run top to bottom.**

| arm | train pairs | composition |
|---|---|---|
| `R0` | 804 | 804 real fold-1 train pairs |
| `R1` | 1608 | the same 804 real + 804 synthetic from `fake_pairs_R1/` |

Validation is the **136 real fold-1 tiles (408 images) in both arms**, never a synthetic pixel.

This is the structural twin of `geonusaf-stage4-segformer-b0-R0R1.ipynb` — same 25 sections, same
config dataclass, same asserts, same persistence, same paired bootstrap. **Only section 13 is
genuinely different**, and it is the part-1 SegNeXt model cell carried over verbatim.

## Four deliberate departures from `4__segnext.ipynb`

**1. Step-based, not epoch-based.** R1's training set is twice R0's, so a pass is 100 steps against
R0's 50. Holding `MAX_EPOCHS` fixed would hand R1 twice the gradient updates and the result would
measure compute, not data. Both arms get `MAX_STEPS = 6000` with a 500-step warmup and a cosine
horizon of 6000 — which at R0 is *exactly* part 1's 120-epoch cosine (120 x 50 = 6000). Validation
every 200 steps, patience 8 validations.

**2. The augmentation bug is fixed.** Part-1 SegNeXt seeded a fresh `RandomState` from `self.epoch`
inside `__getitem__`, the main process set `train_ds.epoch` at the top of `train_one_epoch`, and the
train loader ran `persistent_workers=True`. Persistent workers hold their own copy of the dataset
from the first iterator, so `self.epoch` never crossed the process boundary and **every epoch
replayed identical rot90 and jitter draws**. This is the same defect found in U-Net, SegFormer-B0
and TC-SegFormer. Augmentation here draws from the worker-local `np.random` stream, which advances
on every call regardless of worker lifetime. Section 11 asserts it.

**3. Determinism off.** `use_deterministic_algorithms(True)` forces a sort-based backward for the
bilinear upsample, which part 1 measured at roughly 6x on this workload — and SegNeXt pays it worse
than B0 because its head upsamples 8x rather than 4x. Both arms run without it, so the comparison is
unaffected and a fold that cost 2.5-6 h in part 1 costs 1.5-2.5 h here.

**4. `MIN_DELTA` dropped.** Part-1 SegNeXt used `MIN_DELTA=0.0` where the other baselines used
0.002. All three Stage-4 architectures share one stopping rule instead: patience 8 validations on
any improvement.

Consequence: **the R0 number this notebook produces will not equal the published part-1 fold-1
SegNeXt number.** That is the point. R0 and R1 must come out of the same harness or the delta means
nothing. Do not mix the part-1 number into the Stage-4 table.

## One thing this model does that the others do not

The LightHam decoder redraws its NMF bases from `torch.rand` **on every forward pass**
(`MD_RAND_INIT=True`, the published behaviour). Two evaluations of identical weights therefore
differ slightly, which matters more here than it did in part 1: the Stage-4 headline is a
*difference* between two runs, and part of that difference is evaluation noise rather than data.
Section 20 measures it directly — `EVAL_REPEATS` full validation passes on the frozen best
checkpoint — and section 22 prints the R1 - R0 delta against that noise floor. If the delta is
inside it, say so.

## Prerequisites
- Internet **ON**. Secrets: `HF_TOKEN` (write) and `WANDB_API_KEY`. Internet is also what fetches
  the MSCAN-T ImageNet weights; without it, attach `mscan_t_20230227-119e8c9f.pth` as a Kaggle
  dataset and section 13 will find it.
- Inputs attached: `sugam24/satellite-datasets` **and** `sugam24/geonusaf-fake-pairs-block-fold1`.
- GPU T4 x1. Budget ~2-3 h per arm; resume from `last.pt` is automatic if a session dies.

## 1. Environment

Nothing model-specific to install. SegNeXt's official release depends on `mmcv-full` +
`mmsegmentation 0.24`, which fights Kaggle's preinstalled torch; section 13 sidesteps that by
carrying the model source inline, exactly as part 1 did. Only `ptflops` is missing from the image.

`torch_load` is needed here and was not in the B0 Stage-4 notebook: the openmmlab MSCAN checkpoint
carries an `argparse.Namespace` in its `meta` block, which `weights_only=True` refuses.

In [ ]:
!pip install -q ptflops==0.7.3 2>&1 | tail -2

import os, re, gc, json, time, math, random, hashlib, warnings, itertools, copy
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs

def torch_load(path, map_location="cpu"):
    """torch>=2.6 defaults weights_only=True, which rejects the argparse.Namespace that
    mmseg/openmmlab checkpoints carry in their `meta` block. Try the safe path first and
    fall back only when the safe unpickler refuses -- never silently, so a genuinely
    corrupt file still raises."""
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except Exception as e:
        if "WeightsUnpickler" not in str(e) and "weights_only" not in str(e).lower():
            raise
        print(f"  (weights_only=True refused {Path(path).name}: falling back -- "
              f"trusted source, tensors are extracted and the rest discarded)")
        return torch.load(path, map_location=map_location, weights_only=False)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration

`ARM` is the only field you change between sessions. Everything else is identical across the two
runs by construction — that is what makes the delta attributable to the data.

The schedule is expressed in **steps**. `COS_HORIZON_STEPS == MAX_STEPS` so the cosine lands at zero
exactly when training ends; running past it at LR ~ 0 buys nothing.

The architecture block holds the published MSCAN-T settings, unchanged from part 1. `FUSE_STAGE1`
and `MD_RAND_INIT` are ablation switches, not protocol knobs — leave them at the defaults, and if
you ever flip one, flip it in **both** arms or the comparison is void.

`LR_DECODER=6e-4 / LR_ENCODER=6e-5` is 10x the B0 notebook's absolute scale. That is deliberate and
carried over from part 1: it is SegNeXt's own ADE20K recipe (`AdamW(lr=6e-5)` with head
`lr_mult=10`), on the same footing as UNetFormer's 6e-4.

In [ ]:
@dataclass
class CFG:
    # ---- THE ONLY FIELD YOU CHANGE BETWEEN SESSIONS ----
    ARM: str = "R0"                    # "R0" (real only) | "R1" (real + 804 synthetic)

    # ---- run selection: fixed, the generators are fold-1 block models ----
    SPLIT_MODE: str = "block"
    FOLD: int = 1

    # ---- data ----
    DATA_ROOT: Optional[str] = None    # None -> auto-detect under /kaggle/input
    FAKE_ROOT: Optional[str] = None    # None -> auto-detect fake_pairs_R1 under /kaggle/input
    CACHE_DIR: str = "/kaggle/working/cache512"
    PREBUILT_CACHE: Optional[str] = None
    IMG_SIZE: int = 512
    N_CLASSES: int = 6
    IGNORE_INDEX: int = 255
    PRELOAD_RAM: bool = True           # ~1.3 GB real + ~0.9 GB synthetic

    # ---- split ----
    N_FOLDS: int = 3
    BLOCK_BUFFER: int = 2
    # sha1 of the fold-1 train id list, printed by Stage 1/2/3 section 8. Asserted, not eyeballed:
    # if S4 trains on a different real split than the generators saw, the whole experiment is void.
    UPSTREAM_TRAIN_SHA1: str = "eb0aebdd991971c319fc5fde92f8ad4aa3d82b65"

    # ---- synthetic mixing ----
    N_FAKE: int = 804                  # size of fake_pairs_R1, asserted on load
    # "real"  -> class weights from the 804 real pairs, IDENTICAL in both arms (default)
    # "mixed" -> recomputed over real+fake, which makes the loss a second moving variable
    WEIGHTS_FROM: str = "real"
    # Real records already carry hf/vf as separate rows; synthetic pairs do not. Leaving this off
    # keeps the augmentation policy per-record and identical for every training row, which is the
    # cleaner protocol claim. Turning it on gives synthetic pairs the same flip coverage instead.
    AUG_FLIP_FAKE: bool = False

    # ---- model: MSCAN-T (published tiny config, unchanged from part 1) ----
    EMBED_DIMS: tuple = (32, 64, 160, 256)
    MLP_RATIOS: tuple = (8, 8, 4, 4)
    DEPTHS: tuple = (3, 3, 5, 2)
    DROP_PATH_RATE: float = 0.1
    DROP_RATE: float = 0.0
    HEAD_CHANNELS: int = 256
    HAM_CHANNELS: int = 256
    MD_R: int = 16                     # NMF rank; the tiny config's ham_kwargs value
    MD_TRAIN_STEPS: int = 6
    MD_EVAL_STEPS: int = 7
    MD_RAND_INIT: bool = True          # published behaviour; False = fixed bases (ablation only)
    GN_GROUPS: int = 32                # decoder uses GroupNorm, not BatchNorm
    CLASSIFIER_DROPOUT: float = 0.1
    FUSE_STAGE1: bool = False          # False = stages 2-4 @ 1/8 (published). True = 1/4 ablation.

    # ---- pretrained encoder: identical init in both arms ----
    PRETRAINED_URL: str = ("https://download.openmmlab.com/mmsegmentation/v0.5/"
                           "pretrain/segnext/mscan_t_20230227-119e8c9f.pth")
    PRETRAINED_LOCAL: Optional[str] = None   # None -> also rglob'd from /kaggle/input
    PRETRAINED_REQUIRED: bool = True
    PRETRAINED_MIN_COVERAGE: float = 0.98

    # ---- optimization: STEP-based, so R0 and R1 get identical compute ----
    BATCH_SIZE: int = 16
    GRAD_ACCUM: int = 1                # asserted == 1 below; see the note
    LR_DECODER: float = 6e-4           # SegNeXt ADE20K recipe: base 6e-5 with head lr_mult=10
    LR_ENCODER: float = 6e-5           # 10x differential, per protocol
    WEIGHT_DECAY: float = 0.01
    NO_DECAY_ON_NORM_BIAS: bool = True
    MAX_STEPS: int = 6000              # == 120 epochs at R0's 50 steps/pass
    COS_HORIZON_STEPS: int = 6000
    WARMUP_STEPS: int = 500
    VAL_EVERY_STEPS: int = 200
    PATIENCE_VALS: int = 8             # 8 x 200 = 1600 steps without improvement
    CLIP_NORM: float = 1.0
    LAMBDA_LOVASZ: float = 0.75
    EN_BETA: float = 0.9999
    WEIGHT_CAP: float = 10.0
    LABEL_SMOOTHING: float = 0.05
    USE_EMA: bool = True
    EMA_DECAY: float = 0.999

    # ---- augmentation ----
    AUG_ROT90: bool = True
    AUG_PHOTOMETRIC: bool = True
    AUG_BRIGHTNESS: float = 0.10
    AUG_CONTRAST: float = 0.10

    # ---- evaluation ----
    BOOTSTRAP_N: int = 2000            # resamples over the 136 val base tiles
    EVAL_REPEATS: int = 3              # full val passes on the frozen best model; >1 measures the
                                       # NMF re-sampling noise floor, which section 22 needs

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 4
    AMP: bool = True
    MAX_HOURS: float = 11.0            # stop and push state before Kaggle's 12 h cap

    # ---- persistence ----
    HF_USER: str = "sugam24"
    HF_PREFIX: str = "geonusaf-s4-segnext-t"
    PUSH_LAST_EVERY_VALS: int = 5      # full-state pushes, in validations
    WANDB_PROJECT: str = "geonusaf-stage4"
    OUT_DIR: str = "/kaggle/working/out"

    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
assert cfg.ARM in ("R0", "R1"), cfg.ARM
# The step budget counts LOOP iterations. With GRAD_ACCUM > 1 the cosine horizon would be
# COS_HORIZON_STEPS/GRAD_ACCUM optimizer steps and the "6000 == part 1's 120 epochs" identity
# would quietly stop holding. If VRAM is tight, see the checklist -- do not reach for accumulation.
assert cfg.GRAD_ACCUM == 1, "step-based schedule assumes one optimizer step per loop iteration"
USE_FAKE = (cfg.ARM == "R1")

RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.ARM}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
tag = f"{cfg.ARM}_{cfg.SPLIT_MODE}_fold{cfg.FOLD}"

print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)
print(f"arm {cfg.ARM}: real 804 + synthetic {cfg.N_FAKE if USE_FAKE else 0}")
print(f"arch: MSCAN-T dims={cfg.EMBED_DIMS} depths={cfg.DEPTHS} | ham R={cfg.MD_R} "
      f"| fuse_stage1={cfg.FUSE_STAGE1} | md_rand_init={cfg.MD_RAND_INIT}")
print(f"opt : lr {cfg.LR_DECODER}/{cfg.LR_ENCODER} wd={cfg.WEIGHT_DECAY} "
      f"drop_path={cfg.DROP_PATH_RATE} smooth={cfg.LABEL_SMOOTHING} ema={cfg.USE_EMA}")
print(f"schedule: {cfg.MAX_STEPS} steps | warmup {cfg.WARMUP_STEPS} | cosine to "
      f"{cfg.COS_HORIZON_STEPS} | val every {cfg.VAL_EVERY_STEPS} | patience "
      f"{cfg.PATIENCE_VALS} vals")

## 3. Determinism — deliberately OFF

Part 1 ran `use_deterministic_algorithms(True, warn_only=True)` under protocol 6.8 and paid for it:
that setting forces a sort-based `indexing_backward_kernel` for the bilinear-upsample backward,
which part 1's own profiling put at ~63% of all CUDA time. SegNeXt upsamples logits 8x rather than
B0's 4x, so it pays the most of any model in the table.

It buys nothing here. Both arms share the seed, the split, the initialisation and the schedule, so
run-to-run kernel nondeterminism is common-mode noise on a *comparison*, not a confound.
`cudnn.benchmark` is on instead, since the input shape never varies.

The NMF bases are a separate, architectural source of nondeterminism that no torch flag removes —
see the title cell and section 20.

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def worker_init_fn(worker_id):
    # each worker gets its OWN np.random stream, and that stream advances on every augmentation
    # draw. This is what makes augmentation live across passes -- see section 11.
    s = (cfg.SEED * 7919 + worker_id) % (2**31)
    np.random.seed(s); random.seed(s)

set_seed(cfg.SEED)
g = torch.Generator(); g.manual_seed(cfg.SEED)
print("seeded:", cfg.SEED, "| deterministic algorithms: OFF | cudnn.benchmark: ON")

## 4. Filename parsing and base-tile grouping

Verbatim from the part-1 baseline, including the 1224 / 408 asserts. `DATA_ROOT` is auto-detected
because the Kaggle mount path differs between accounts and hard-coding it has already cost a
session once.

In [ ]:
def find_data_root(explicit):
    if explicit and Path(explicit, "images").is_dir(): return explicit
    for p in Path("/kaggle/input").rglob("satellite_dataset"):
        if (p / "images").is_dir() and (p / "masks").is_dir(): return str(p)
    raise FileNotFoundError("satellite_dataset not found -- attach sugam24/satellite-datasets")

DATA_ROOT = find_data_root(cfg.DATA_ROOT)
IMG_DIR  = Path(DATA_ROOT) / "images"
MASK_DIR = Path(DATA_ROOT) / "masks"
print("DATA_ROOT:", DATA_ROOT)

PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")
records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert all(len(v) == 3 for v in groups.values())
print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")

## 5. Mask colour diagnostic

The channel-order guard. If BGR/RGB ever inverted, Residential and River would swap and every
number below would be wrong in a way that looks plausible.

In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0, (0, 128, 0): 1, (0, 0, 128): 2,
    (0, 128, 128): 3, (128, 128, 0): 4, (128, 0, 128): 5,
}

def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

LUT = np.full(1 << 24, 255, dtype=np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    LUT[pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0]] = idx

rng = np.random.RandomState(cfg.SEED)
sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]
class_px = np.zeros(7, dtype=np.int64)
for _, _, _, mname in sample:
    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    idx = LUT[pack_bgr(m)]
    class_px += np.bincount(np.where(idx == 255, 6, idx).ravel(), minlength=7)

tot = class_px.sum()
for i, nm in enumerate(list(cfg.CLASS_NAMES) + ["<ignore>"]):
    print(f"  {i if i < 6 else 255:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%")
assert class_px[0] > class_px[2] * 3, "Residential <= River -- channel order inverted. STOP."
print("\nchannel-order check PASSED")

## 6. One-time repack to a 512 cache

Unchanged. If you already saved this cache as a Kaggle dataset from a part-1 run, point
`PREBUILT_CACHE` at it and this becomes a no-op — it saves ~15 min per session and there are two
sessions here.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)
MARKER = CACHE / "_complete.json"

def _repack_one(rec):
    _, _, iname, mname = rec
    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    lab = LUT[pack_bgr(msk)]
    lab = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)

if MARKER.exists():
    print(f"cache present at {CACHE} ({json.load(open(MARKER))['n']} pairs) -- skipping repack")
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    (CACHE / "images").mkdir(parents=True, exist_ok=True)
    (CACHE / "labels").mkdir(parents=True, exist_ok=True)
    t0 = time.time(); counts = {}
    with ProcessPoolExecutor(max_workers=4) as ex:
        for rec, b in zip(records, ex.map(_repack_one, records, chunksize=8)):
            counts[rec[2]] = b.tolist()
            if len(counts) % 200 == 0:
                el = time.time() - t0
                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  "
                      f"eta {el/len(counts)*(len(records)-len(counts)):.0f}s")
    json.dump({"n": len(counts), "counts": counts}, open(MARKER, "w"))
    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")

PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in json.load(open(MARKER))["counts"].items()}

## 7. Load the real cache into RAM

`RAM` is keyed by image filename and is the single lookup table the Dataset reads from. Section 9
adds the synthetic pairs to this **same** dict, so the Dataset needs no branch and no adapter.

In [ ]:
RAM = {}
if cfg.PRELOAD_RAM:
    t0 = time.time()
    for i, (_, _, iname, mname) in enumerate(records):
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        RAM[iname] = (img, lab)
        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")
    mb = sum(a.nbytes + b.nbytes for a, b in RAM.values()) / 1e6
    print(f"preloaded {len(RAM)} real pairs, {mb:.0f} MB, {time.time()-t0:.0f}s")
    u = np.unique(np.concatenate([b[::32, ::32].ravel() for _, b in RAM.values()]))
    assert set(u.tolist()) <= set(range(6)) | {255}, f"unexpected label values: {u}"
    print("label value check PASSED:", u.tolist())

## 8. Split, and the four-way agreement with the generators

`block_split` is verbatim from part 1 and from Stages 1-3. Unlike TC-SegFormer, part-1 SegNeXt used
**this** splitter, so its fold-1 train set is the same 268 tiles the generators saw — but the sha1
is still **asserted**, not printed for eyeballing: Stage 1, Stage 2 and Stage 3 all trained or
sampled on this exact list, and if Stage 4's real half were a different split the synthetic pairs
would have been generated from tiles that are now in validation. That is the leakage failure this
experiment cannot survive, and it would be invisible in every metric.

In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)

def expand(ids): return [r for i in ids for r in groups[i]]
real_tr_recs, va_recs = expand(tr_ids), expand(va_ids)

assert not (set(tr_ids) & set(va_ids)), "base-tile leakage between train and val"
split_sha1 = hashlib.sha1(",".join(map(str, sorted(tr_ids))).encode()).hexdigest()
assert split_sha1 == cfg.UPSTREAM_TRAIN_SHA1, (
    "SPLIT DISAGREEMENT with the generators -- the synthetic pairs were made from a different "
    f"train set.\n  upstream: {cfg.UPSTREAM_TRAIN_SHA1}\n  here    : {split_sha1}")

print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_ids)} train tiles / {len(va_ids)} val tiles")
print(f"  -> {len(real_tr_recs)} real train pairs / {len(va_recs)} val images")
print(f"  train_sha1 {split_sha1[:12]} -- four-way agreement with S1/S2/S3 ASSERTED")
assert len(real_tr_recs) == 804, len(real_tr_recs)

## 9. The synthetic half

Loaded only when `ARM == "R1"`. The Stage-3 contract is re-asserted here rather than trusted:
804 pairs, indices 0..803 complete, images 512x512x3 `uint8` BGR, masks 512x512 `uint8` with
values in {0..5, 255}. Same dtypes and channel order as the real cache, so they go into the same
`RAM` dict and the Dataset cannot tell them apart.

Synthetic records carry `base_id = -(i+1)`. Negative ids can never collide with a real tile, which
means the assert at the end of this section — that no synthetic record reached validation — is
checking something real rather than restating an assumption.

The printed class-share table is worth reading before you train. The synthetic half carries River
in roughly twice as many masks as the real half does; that is the intended effect and it is also
the thing most likely to shift the confusion matrix.

In [ ]:
def find_fake_root(explicit, name="fake_pairs_R1"):
    if explicit and Path(explicit).is_dir(): return explicit
    for p in Path("/kaggle/input").rglob(name):
        if p.is_dir() and any(p.glob("genimage*.png")): return str(p)
    raise FileNotFoundError(f"{name} not found -- attach sugam24/geonusaf-fake-pairs-block-fold1")

fake_tr_recs = []
if USE_FAKE:
    FAKE_ROOT = Path(find_fake_root(cfg.FAKE_ROOT))
    print("FAKE_ROOT:", FAKE_ROOT)

    idxs = sorted(int(p.stem[len("genimage"):]) for p in FAKE_ROOT.glob("genimage*.png"))
    assert len(idxs) == cfg.N_FAKE, f"{len(idxs)} synthetic images, expected {cfg.N_FAKE}"
    assert idxs == list(range(cfg.N_FAKE)), "synthetic index range is not 0..n-1"

    t0 = time.time(); fake_px = np.zeros(7, dtype=np.int64)
    for i in idxs:
        iname, mname = f"genimage{i}.png", f"genmask{i}.png"
        img = cv2.imread(str(FAKE_ROOT / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(FAKE_ROOT / mname), cv2.IMREAD_GRAYSCALE)
        assert img is not None and lab is not None, f"unreadable synthetic pair {i}"
        assert img.shape == (cfg.IMG_SIZE, cfg.IMG_SIZE, 3) and img.dtype == np.uint8, \
            f"synthetic image {i} is {img.shape} {img.dtype}"
        assert lab.shape == (cfg.IMG_SIZE, cfg.IMG_SIZE) and lab.dtype == np.uint8, \
            f"synthetic mask {i} is {lab.shape} {lab.dtype}"
        b = np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)
        assert b.shape[0] == 7, f"synthetic mask {i} has values outside the contract"
        RAM[iname] = (img, lab)
        PIX_COUNTS[iname] = b
        fake_px += b
        fake_tr_recs.append((-(i + 1), "fake", iname, mname))
        if (i + 1) % 200 == 0: print(f"  loaded {i+1}/{cfg.N_FAKE}")
    print(f"loaded {len(fake_tr_recs)} synthetic pairs in {time.time()-t0:.0f}s")

    real_px = np.zeros(7, dtype=np.int64)
    for _, _, iname, _ in real_tr_recs: real_px += PIX_COUNTS[iname]
    print(f"\n{'class':<14}{'real train %':>14}{'synthetic %':>13}{'ratio':>8}")
    for i, nm in enumerate(list(cfg.CLASS_NAMES) + ["<ignore>"]):
        r_, f_ = real_px[i]/real_px.sum()*100, fake_px[i]/fake_px.sum()*100
        print(f"{nm:<14}{r_:13.3f}%{f_:12.3f}%{(f_/max(r_,1e-9)):8.2f}")

tr_recs = real_tr_recs + fake_tr_recs
FAKE_NAMES = {r[2] for r in fake_tr_recs}
assert not (FAKE_NAMES & {r[2] for r in va_recs}), "SYNTHETIC PAIR IN VALIDATION -- stop"
assert all(r[0] > 0 for r in va_recs), "negative base id in validation"
print(f"\narm {cfg.ARM}: {len(tr_recs)} train pairs "
      f"({len(real_tr_recs)} real + {len(fake_tr_recs)} synthetic) | {len(va_recs)} val images")

## 10. Effective-number class weights

Computed from the **real** train pairs only, so `CLASS_W` is bit-identical in both arms and the
loss is not a second moving variable. `WEIGHTS_FROM="mixed"` recomputes over real+synthetic
instead; both are printed either way so the paper can state what the choice cost.

In [ ]:
EN_BETA_AUTO = True

def effective_number_weights(recs, cap, n_classes, beta_fixed, auto):
    n = np.zeros(n_classes, dtype=np.float64)
    for _, _, iname, _ in recs:
        n += PIX_COUNTS[iname][:n_classes]
    beta = (1.0 - 1.0 / n.sum()) if auto else beta_fixed
    eff = (1.0 - np.power(beta, n)) / (1.0 - beta)
    w = 1.0 / np.maximum(eff, 1e-12)
    w = w / w.mean()
    w = np.minimum(w, cap)
    return w / w.mean(), n, beta

W_REAL, PX_REAL, BETA_R = effective_number_weights(
    real_tr_recs, cfg.WEIGHT_CAP, cfg.N_CLASSES, cfg.EN_BETA, EN_BETA_AUTO)
W_MIXED, PX_MIXED, BETA_M = effective_number_weights(
    tr_recs, cfg.WEIGHT_CAP, cfg.N_CLASSES, cfg.EN_BETA, EN_BETA_AUTO)

CLASS_W = W_REAL if cfg.WEIGHTS_FROM == "real" else W_MIXED
print(f"WEIGHTS_FROM = {cfg.WEIGHTS_FROM}\n")
print(f"{'class':<14}{'train px %':>12}{'w(real)':>10}{'w(mixed)':>11}")
for i, nm in enumerate(cfg.CLASS_NAMES):
    print(f"{nm:<14}{PX_REAL[i]/PX_REAL.sum()*100:11.2f}%{W_REAL[i]:10.3f}{W_MIXED[i]:11.3f}")
print(f"weight ratio max/min: real {W_REAL.max()/W_REAL.min():.1f} | "
      f"mixed {W_MIXED.max()/W_MIXED.min():.1f}")

CLASS_W_T = torch.tensor(CLASS_W, dtype=torch.float32).cuda()

## 11. Dataset and DataLoaders — with the augmentation bug fixed

Part 1 seeded a fresh `RandomState` from `(SEED, self.epoch, index)` and relied on the main process
setting `train_ds.epoch` at the top of every epoch. With `persistent_workers=True` that assignment
never crossed the process boundary, so the augmentation was frozen after the first epoch. Here the
draws come from the worker-local `np.random` stream, which `worker_init_fn` seeds once per worker
and which then advances on every single call — no epoch bookkeeping to propagate, nothing to go
stale. It also has to work this way now: a step-based loop has no epoch counter to set.

The assert below draws the same record twice and requires the results to differ. It runs in the
main process, so it tests the mechanism rather than the multiprocessing path, but the mechanism is
exactly what was broken.

Note for the paper, unchanged from part 1: MSCA's strip convolutions are axis-aligned (1xk and kx1),
so rot90 is doing more work for this architecture than for the isotropic ones — it is what teaches
the horizontal and vertical branches to cover both road orientations. Which makes the frozen-
augmentation bug worse here than elsewhere, and is one more reason the part-1 number is not a usable
R0.

Synthetic and real records are read through the same path. `AUG_FLIP_FAKE` is the one place they
can differ, and it is off by default — see section 2.

In [ ]:
IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)

class SatSeg(Dataset):
    def __init__(self, recs, train: bool):
        self.recs, self.train = recs, train

    def __len__(self): return len(self.recs)

    def _read(self, iname, mname):
        if cfg.PRELOAD_RAM:
            img, lab = RAM[iname]
            return img.copy(), lab.copy()
        if iname in FAKE_NAMES:
            img = cv2.imread(str(FAKE_ROOT / iname), cv2.IMREAD_COLOR)
            lab = cv2.imread(str(FAKE_ROOT / mname), cv2.IMREAD_GRAYSCALE)
        else:
            img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
            lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        return img, lab

    def __getitem__(self, i):
        _, variant, iname, mname = self.recs[i]
        img, lab = self._read(iname, mname)

        if self.train:
            if cfg.AUG_FLIP_FAKE and variant == "fake":
                # real records carry hf/vf as separate rows; this gives synthetic ones the same
                # flip coverage instead of leaving them with only rot90
                f = np.random.randint(3)
                if f == 1: img, lab = img[:, ::-1], lab[:, ::-1]
                elif f == 2: img, lab = img[::-1], lab[::-1]
            if cfg.AUG_ROT90:
                k = np.random.randint(4)
                if k: img, lab = np.rot90(img, k), np.rot90(lab, k)
            img = np.ascontiguousarray(img); lab = np.ascontiguousarray(lab)
            if cfg.AUG_PHOTOMETRIC:
                a = 1.0 + np.random.uniform(-cfg.AUG_CONTRAST, cfg.AUG_CONTRAST)
                b = 255.0 * np.random.uniform(-cfg.AUG_BRIGHTNESS, cfg.AUG_BRIGHTNESS)
                img = np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8)

        x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        x = (x - IMNET_MEAN) / IMNET_STD
        x = torch.from_numpy(x.transpose(2, 0, 1))
        y = torch.from_numpy(np.ascontiguousarray(lab).astype(np.int64))
        return x, y

train_ds, val_ds = SatSeg(tr_recs, True), SatSeg(va_recs, False)

# the bug check: two draws of the same record must differ
np.random.seed(0)
_a, _ = train_ds[0]
_b, _ = train_ds[0]
assert not torch.allclose(_a, _b), (
    "augmentation is STATIC -- two draws of record 0 are identical. This is the "
    "persistent_workers bug from the part-1 notebook and it must not ship.")
print("augmentation liveness check PASSED (two draws of record 0 differ)")

train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,
                      worker_init_fn=worker_init_fn, generator=g)
val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True, persistent_workers=True,
                      worker_init_fn=worker_init_fn)

steps_per_pass = len(train_ld)
xb, yb = next(iter(train_ld))
print(f"{xb.shape} {xb.dtype} | {yb.shape} {yb.dtype}")
print(f"steps per pass: {steps_per_pass} | {cfg.MAX_STEPS} steps = "
      f"{cfg.MAX_STEPS/steps_per_pass:.1f} passes over the training set")

## 12. Loss — weighted CE + Lovasz-Softmax

Unchanged from part 1, including the fp32 cast outside autocast and the present-classes-only
Lovasz. `CLASS_W_T` comes from section 10 and is identical in both arms.

In [ ]:
def lovasz_grad(gt_sorted):
    p = len(gt_sorted); gts = gt_sorted.sum()
    inter = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jac = 1.0 - inter / union
    if p > 1: jac[1:p] = jac[1:p] - jac[0:-1]
    return jac

def lovasz_softmax_flat(probas, labels, n_classes):
    if probas.numel() == 0: return probas.sum() * 0.0
    losses = []
    for c in range(n_classes):
        fg = (labels == c).float()
        if fg.sum() == 0: continue
        err = (fg - probas[:, c]).abs()
        err_sorted, perm = torch.sort(err, 0, descending=True)
        losses.append(torch.dot(err_sorted, lovasz_grad(fg[perm])))
    return torch.stack(losses).mean() if losses else probas.sum() * 0.0

class ComboLoss(nn.Module):
    def __init__(self, weights, lam, ignore, n_classes, smooth=0.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weights, ignore_index=ignore, label_smoothing=smooth)
        self.lam, self.ignore, self.nc = lam, ignore, n_classes

    def forward(self, logits, target):
        logits = logits.float()
        ce = self.ce(logits, target)
        B, C, H, W = logits.shape
        p = logits.permute(0, 2, 3, 1).reshape(-1, C).softmax(1)
        t = target.reshape(-1)
        keep = t != self.ignore
        lv = lovasz_softmax_flat(p[keep], t[keep], self.nc)
        return ce + self.lam * lv, ce.detach(), lv.detach()

criterion = ComboLoss(CLASS_W_T, cfg.LAMBDA_LOVASZ, cfg.IGNORE_INDEX,
                      cfg.N_CLASSES, cfg.LABEL_SMOOTHING).cuda()
print("loss ready | lambda =", cfg.LAMBDA_LOVASZ, "| label_smoothing =", cfg.LABEL_SMOOTHING)

## 13. Model, optimizer, step-based scheduler

**This is the only section that differs substantively from the Stage-4 B0 notebook**, and it is
part-1 section 12 carried over whole.

### Why the model is a source string
The official release depends on `mmcv-full` + `mmsegmentation 0.24.1`, both of which pin CUDA/torch
combinations Kaggle's image does not match, so the ~270 lines of actual model code are reproduced
inline with the mmcv wrappers replaced. Module and parameter names follow the official repo exactly,
so the published ImageNet checkpoint loads without a conversion table. It is held as a **string**
because `SegformerForSemanticSegmentation` can rebuild itself from `transformers` and this class
cannot: the definition has to travel to Hugging Face alongside the weights or the checkpoint is dead
on arrival. One copy of the bytes, written to `segnext_model.py`, then `exec`'d — what trains and
what gets pushed cannot diverge. Source: Visual-Attention-Network/SegNeXt, Apache-2.0.

### What changed for Stage 4
1. **The scheduler horizon is `COS_HORIZON_STEPS` straight from the config**, not
   `steps_per_epoch * COS_HORIZON_EPOCHS`. That single line is what makes the two arms
   compute-matched — under the old form, R1's doubled `steps_per_pass` would have stretched the
   cosine to 12000 steps and handed it twice the schedule.
2. **`ARCH_SIG`**, a sha1 over the model source and the `ARCH` dict, is written into both
   checkpoints and asserted on resume. Two arms that disagree here are not comparable, and a resume
   that silently loads a differently-shaped run is worse than a crash.

Everything else — MSCA, the Hamburger, the fp32 NMF, the one-step gradient, the pretrained loader,
the four parameter groups, the LRs — is byte-comparable with part 1. Both arms start from the same
ImageNet init and the coverage assert is what guarantees it.

### What the sanity prints must show
- `params: 4.23 M` — a different number means the config is wrong.
- `pretrained: loaded N/N backbone tensors (100.0%)`. Below `PRETRAINED_MIN_COVERAGE` this raises;
  paste the printed missing/unexpected sample rather than setting `PRETRAINED_REQUIRED=False`.
- `head fuses at stride 8 -> logits 64x64, upsampled 8x to 512` — the coarse-head caveat, printed
  so it lands in the run log. A 12 px road is ~1.4 px wide in that logit grid, which is the standing
  suspect if Road and River underperform.
- `logits after upsample: (1, 6, 512, 512)`.

In [ ]:
# The model is defined as a source string, written to disk, and then exec'd.
# One copy of the code: what runs and what gets pushed to HF cannot diverge, and
# there is no inspect.getsource step to fail silently.
SEGNEXT_SRC = r'''
# ---------------------------------------------------------------------------
# SegNeXt-T = MSCAN-T encoder + LightHam decoder.
# Ported from https://github.com/Visual-Attention-Network/SegNeXt (Apache-2.0),
# mmcv/mmsegmentation dependencies removed. Parameter names match the original
# so the published ImageNet-1K MSCAN checkpoints load directly.
#
# This file is the architecture definition that travels with a GeoNUSAF SegNeXt-T
# checkpoint. Rebuild with:  SegNeXt(torch.load('best.pt')['arch'])
# ---------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F


class DropPath(nn.Module):
    """Per-sample stochastic depth."""
    def __init__(self, p=0.0):
        super().__init__()
        self.p = p
    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep


class ConvModule(nn.Module):
    """mmcv.cnn.ConvModule without mmcv: conv -> optional GroupNorm -> optional ReLU.
    mmcv's bias='auto' means bias=False whenever a norm follows; reproduced here."""
    def __init__(self, cin, cout, k=1, norm=False, act=False, gn_groups=32):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, k, bias=not norm)
        self.norm = nn.GroupNorm(gn_groups, cout) if norm else None
        self.act = nn.ReLU(inplace=True) if act else None
    def forward(self, x):
        x = self.conv(x)
        if self.norm is not None:
            x = self.norm(x)
        if self.act is not None:
            x = self.act(x)
        return x


class Mlp(nn.Module):
    """1x1 -> depthwise 3x3 -> GELU -> 1x1. Conv-based FFN, no LayerNorm."""
    def __init__(self, dim, hidden, drop=0.0):
        super().__init__()
        self.fc1 = nn.Conv2d(dim, hidden, 1)
        self.dwconv = nn.Conv2d(hidden, hidden, 3, 1, 1, bias=True, groups=hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Conv2d(hidden, dim, 1)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        x = self.drop(self.act(self.dwconv(self.fc1(x))))
        return self.drop(self.fc2(x))


class MSCA(nn.Module):
    """Multi-Scale Convolutional Attention. Three parallel depthwise strip-conv
    branches (7 / 11 / 21) build an attention map that multiplies the input.
    The strip shape is what makes this architecture worth testing on 8-11 px
    roads and rivers: long along the structure, 1 px across it."""
    def __init__(self, dim):
        super().__init__()
        self.conv0   = nn.Conv2d(dim, dim, 5, padding=2, groups=dim)
        self.conv0_1 = nn.Conv2d(dim, dim, (1, 7),  padding=(0, 3),  groups=dim)
        self.conv0_2 = nn.Conv2d(dim, dim, (7, 1),  padding=(3, 0),  groups=dim)
        self.conv1_1 = nn.Conv2d(dim, dim, (1, 11), padding=(0, 5),  groups=dim)
        self.conv1_2 = nn.Conv2d(dim, dim, (11, 1), padding=(5, 0),  groups=dim)
        self.conv2_1 = nn.Conv2d(dim, dim, (1, 21), padding=(0, 10), groups=dim)
        self.conv2_2 = nn.Conv2d(dim, dim, (21, 1), padding=(10, 0), groups=dim)
        self.conv3   = nn.Conv2d(dim, dim, 1)
    def forward(self, x):
        u = x
        a = self.conv0(x)
        # RHS is evaluated against the pre-update `a`, matching the reference implementation
        a = a + self.conv0_2(self.conv0_1(a)) \
              + self.conv1_2(self.conv1_1(a)) \
              + self.conv2_2(self.conv2_1(a))
        return self.conv3(a) * u


class SpatialAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj_1 = nn.Conv2d(dim, dim, 1)
        self.activation = nn.GELU()
        self.spatial_gating_unit = MSCA(dim)
        self.proj_2 = nn.Conv2d(dim, dim, 1)
    def forward(self, x):
        s = x
        x = self.spatial_gating_unit(self.activation(self.proj_1(x)))
        return self.proj_2(x) + s


class MSCABlock(nn.Module):
    """BatchNorm (not LayerNorm) + attention + conv-FFN, both with layer scale."""
    def __init__(self, dim, mlp_ratio, drop=0.0, drop_path=0.0, ls_init=1e-2):
        super().__init__()
        self.norm1 = nn.BatchNorm2d(dim)
        self.attn = SpatialAttention(dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = nn.BatchNorm2d(dim)
        self.mlp = Mlp(dim, int(dim * mlp_ratio), drop)
        self.layer_scale_1 = nn.Parameter(ls_init * torch.ones(dim))
        self.layer_scale_2 = nn.Parameter(ls_init * torch.ones(dim))
    def forward(self, x):
        x = x + self.drop_path(self.layer_scale_1[:, None, None] * self.attn(self.norm1(x)))
        x = x + self.drop_path(self.layer_scale_2[:, None, None] * self.mlp(self.norm2(x)))
        return x


class StemConv(nn.Module):
    """Two stride-2 3x3 convs -> stage 1 at 1/4 resolution."""
    def __init__(self, cin, cout):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(cin, cout // 2, 3, 2, 1), nn.BatchNorm2d(cout // 2), nn.GELU(),
            nn.Conv2d(cout // 2, cout, 3, 2, 1), nn.BatchNorm2d(cout))
    def forward(self, x):
        return self.proj(x)


class OverlapPatchEmbed(nn.Module):
    def __init__(self, cin, cout, k=3, stride=2):
        super().__init__()
        self.proj = nn.Conv2d(cin, cout, k, stride, padding=k // 2)
        self.norm = nn.BatchNorm2d(cout)
    def forward(self, x):
        return self.norm(self.proj(x))


class MSCAN(nn.Module):
    """Four-stage pyramid at 1/4, 1/8, 1/16, 1/32."""
    def __init__(self, embed_dims, mlp_ratios, depths, drop_rate=0.0, drop_path_rate=0.1):
        super().__init__()
        self.depths = list(depths)
        self.embed_dims = list(embed_dims)
        dpr = [float(v) for v in torch.linspace(0, drop_path_rate, max(1, sum(depths)))]
        cur = 0
        for i in range(4):
            pe = StemConv(3, embed_dims[0]) if i == 0 \
                 else OverlapPatchEmbed(embed_dims[i - 1], embed_dims[i])
            blk = nn.ModuleList([MSCABlock(embed_dims[i], mlp_ratios[i], drop_rate, dpr[cur + j])
                                 for j in range(depths[i])])
            cur += depths[i]
            setattr(self, f"patch_embed{i + 1}", pe)
            setattr(self, f"block{i + 1}", blk)
            setattr(self, f"norm{i + 1}", nn.LayerNorm(embed_dims[i]))
    def forward(self, x):
        outs = []
        for i in range(4):
            x = getattr(self, f"patch_embed{i + 1}")(x)
            for b in getattr(self, f"block{i + 1}"):
                x = b(x)
            B, C, H, W = x.shape
            # the stage norm is a LayerNorm over channels -> flatten to tokens and back
            x = getattr(self, f"norm{i + 1}")(x.flatten(2).transpose(1, 2))
            x = x.transpose(1, 2).reshape(B, C, H, W)
            outs.append(x)
        return outs


class NMF2D(nn.Module):
    """The 'ham' in Hamburger: non-negative matrix factorisation as the context
    module (Geng et al. 2021). No learnable parameters. The multiplicative-update
    iterations run under no_grad; only the final coefficient update is
    differentiable, which is the published one-step-gradient scheme."""
    def __init__(self, S=1, R=16, train_steps=6, eval_steps=7, rand_init=True, eps=1e-6):
        super().__init__()
        self.S, self.R = S, R
        self.train_steps, self.eval_steps = train_steps, eval_steps
        self.rand_init, self.eps = rand_init, eps
        self.register_buffer("fixed_bases", torch.empty(0), persistent=False)

    def _bases(self, B, D, x):
        if self.rand_init or self.fixed_bases.numel() == 0:
            b = torch.rand(B * self.S, D, self.R, device=x.device, dtype=x.dtype)
            b = F.normalize(b, dim=1)
            if not self.rand_init:
                self.fixed_bases = b[:self.S].detach()
            else:
                return b
        return self.fixed_bases.repeat(B, 1, 1)

    @torch.no_grad()
    def _step(self, x, bases, coef):
        num = torch.bmm(x.transpose(1, 2), bases)
        den = coef.bmm(bases.transpose(1, 2).bmm(bases))
        coef = coef * num / (den + self.eps)
        num = torch.bmm(x, coef)
        den = bases.bmm(coef.transpose(1, 2).bmm(coef))
        bases = bases * num / (den + self.eps)
        return bases, coef

    def forward(self, x):
        B, C, H, W = x.shape
        D, N = C // self.S, H * W
        xf = x.view(B * self.S, D, N)
        bases = self._bases(B, D, xf)
        with torch.no_grad():
            coef = F.softmax(torch.bmm(xf.transpose(1, 2), bases), dim=-1)
            for _ in range(self.train_steps if self.training else self.eval_steps):
                bases, coef = self._step(xf, bases, coef)
        # one differentiable coefficient update -> gradient reaches x, not the bases
        num = torch.bmm(xf.transpose(1, 2), bases)
        den = coef.bmm(bases.transpose(1, 2).bmm(bases))
        coef = coef * num / (den + self.eps)
        return torch.bmm(bases, coef.transpose(1, 2)).view(B, C, H, W)


class Hamburger(nn.Module):
    def __init__(self, ch, R=16, train_steps=6, eval_steps=7, rand_init=True, gn_groups=32):
        super().__init__()
        self.ham_in  = ConvModule(ch, ch, 1, norm=False, act=False)
        self.ham     = NMF2D(R=R, train_steps=train_steps, eval_steps=eval_steps,
                             rand_init=rand_init)
        self.ham_out = ConvModule(ch, ch, 1, norm=True, act=False, gn_groups=gn_groups)
    def forward(self, x):
        e = F.relu(self.ham_in(x), inplace=True)
        e = self.ham_out(self.ham(e))
        return F.relu(x + e, inplace=True)


class LightHamHead(nn.Module):
    """Concat the selected stages at the coarsest-but-one grid, squeeze, factorise, align, classify."""
    def __init__(self, in_channels, channels, ham_channels, n_classes, dropout=0.1,
                 R=16, train_steps=6, eval_steps=7, rand_init=True, gn_groups=32):
        super().__init__()
        self.squeeze   = ConvModule(sum(in_channels), ham_channels, 1, norm=True, act=True,
                                    gn_groups=gn_groups)
        self.hamburger = Hamburger(ham_channels, R, train_steps, eval_steps, rand_init, gn_groups)
        self.align     = ConvModule(ham_channels, channels, 1, norm=True, act=True,
                                    gn_groups=gn_groups)
        self.dropout   = nn.Dropout2d(dropout)
        self.conv_seg  = nn.Conv2d(channels, n_classes, 1)
    def forward(self, feats):
        size = feats[0].shape[2:]
        x = torch.cat([f if f.shape[2:] == size else
                       F.interpolate(f, size=size, mode="bilinear", align_corners=False)
                       for f in feats], dim=1)
        x = self.squeeze(x)
        # fp32: NMF's multiplicative updates underflow their 1e-6 floor in fp16 and the
        # intermediate bmm products can exceed the fp16 range.
        with torch.cuda.amp.autocast(enabled=False):
            x = self.hamburger(x.float())
        return self.conv_seg(self.dropout(self.align(x)))


class SegNeXt(nn.Module):
    def __init__(self, arch):
        super().__init__()
        self.arch = dict(arch)
        a = self.arch
        self.backbone = MSCAN(a["embed_dims"], a["mlp_ratios"], a["depths"],
                              a["drop_rate"], a["drop_path_rate"])
        self.in_index = list(a["in_index"])
        self.decode_head = LightHamHead(
            [a["embed_dims"][i] for i in self.in_index], a["channels"], a["ham_channels"],
            a["n_classes"], a["dropout_ratio"], a["md_r"], a["md_train_steps"],
            a["md_eval_steps"], a["md_rand_init"], a["gn_groups"])
        self._init_head()
    def _init_head(self):
        for m in self.decode_head.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        nn.init.normal_(self.decode_head.conv_seg.weight, std=0.01)
        nn.init.zeros_(self.decode_head.conv_seg.bias)
    def forward(self, x):
        feats = self.backbone(x)
        return self.decode_head([feats[i] for i in self.in_index])
'''

MODEL_SRC_PATH = f"{cfg.OUT_DIR}/segnext_model.py"
Path(MODEL_SRC_PATH).write_text(SEGNEXT_SRC)
exec(compile(SEGNEXT_SRC, MODEL_SRC_PATH, "exec"), globals())
assert Path(MODEL_SRC_PATH).exists() and "SegNeXt" in globals()
print(f"model source written and loaded: {MODEL_SRC_PATH} "
      f"({len(SEGNEXT_SRC.splitlines())} lines)")

# ---- architecture identity: asserted on resume, recorded in both checkpoints ----
ARCH = dict(
    embed_dims=list(cfg.EMBED_DIMS), mlp_ratios=list(cfg.MLP_RATIOS), depths=list(cfg.DEPTHS),
    drop_rate=cfg.DROP_RATE, drop_path_rate=cfg.DROP_PATH_RATE,
    in_index=[0, 1, 2, 3] if cfg.FUSE_STAGE1 else [1, 2, 3],
    channels=cfg.HEAD_CHANNELS, ham_channels=cfg.HAM_CHANNELS, n_classes=cfg.N_CLASSES,
    dropout_ratio=cfg.CLASSIFIER_DROPOUT, md_r=cfg.MD_R, md_train_steps=cfg.MD_TRAIN_STEPS,
    md_eval_steps=cfg.MD_EVAL_STEPS, md_rand_init=cfg.MD_RAND_INIT, gn_groups=cfg.GN_GROUPS)

ARCH_SIG = hashlib.sha1(
    (SEGNEXT_SRC + json.dumps(ARCH, sort_keys=True)).encode()).hexdigest()[:12]

# ---- pretrained MSCAN-T: identical init in both arms, coverage enforced ----
_ALIASES = [(r"^(module\.|backbone\.)+", ""),        # full-seg checkpoints prefix the backbone
            (r"\.dwconv\.dwconv\.", ".dwconv."),     # original repo wraps dwconv one level deeper
            (r"\.projection\.", ".proj.")]           # mmseg port renamed patch-embed conv

def find_pretrained(explicit):
    if explicit and Path(explicit).exists(): return explicit, "explicit path"
    for p in Path("/kaggle/input").rglob("mscan_t*.pth"):
        return str(p), "attached Kaggle dataset"
    local = f"{cfg.OUT_DIR}/mscan_pretrained.pth"
    if not Path(local).exists():
        torch.hub.download_url_to_file(cfg.PRETRAINED_URL, local)   # needs internet ON
    return local, cfg.PRETRAINED_URL

def load_mscan_pretrained(model):
    path, src = find_pretrained(cfg.PRETRAINED_LOCAL)
    sd = torch_load(path, "cpu")
    for k in ("state_dict", "model", "module"):
        if isinstance(sd, dict) and k in sd and isinstance(sd[k], dict): sd = sd[k]

    renamed = {}
    for k, v in sd.items():
        for pat, rep in _ALIASES: k = re.sub(pat, rep, k)
        if k.split(".")[0] in ("head", "fc", "decode_head", "auxiliary_head"): continue
        renamed[k] = v

    tgt = {k: v for k, v in model.backbone.state_dict().items()
           if not k.endswith("num_batches_tracked")}
    keep  = {k: v for k, v in renamed.items() if k in tgt and tgt[k].shape == v.shape}
    shape = {k: (tuple(v.shape), tuple(tgt[k].shape))
             for k, v in renamed.items() if k in tgt and tgt[k].shape != v.shape}
    missing = [k for k in tgt if k not in keep]
    extra   = [k for k in renamed if k not in tgt]
    model.backbone.load_state_dict(keep, strict=False)

    cov = len(keep) / max(len(tgt), 1)
    print(f"pretrained: loaded {len(keep)}/{len(tgt)} backbone tensors ({cov*100:.1f}%) from {src}")
    if shape:  print("  SHAPE MISMATCH  :", list(shape.items())[:5])
    if missing: print("  MISSING (target):", missing[:8], f"... +{max(0,len(missing)-8)} more")
    if extra:   print("  UNUSED (ckpt)   :", extra[:8], f"... +{max(0,len(extra)-8)} more")
    if cov < cfg.PRETRAINED_MIN_COVERAGE and cfg.PRETRAINED_REQUIRED:
        raise RuntimeError(
            f"only {cov*100:.1f}% of MSCAN-T loaded. Fix the key mapping (paste the MISSING/UNUSED "
            f"samples above) rather than training this backbone from scratch.")
    return cov

def build_model():
    return SegNeXt(ARCH)

def forward_logits(m, x):
    # LightHamHead emits logits at the stride of its first input stage (1/8 by default);
    # upsample to input size so loss and metrics stay at native 512, matching every other model.
    lo = m(x)
    return F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)

model = build_model()
PRETRAIN_COVERAGE = load_mscan_pretrained(model)
model = model.cuda()

def param_groups(m):
    def split(mod):
        dec, nod = [], []
        for n, p in mod.named_parameters():
            if not p.requires_grad: continue
            # ndim<=1 catches BN/LN/GN gains, all biases, and both layer_scale vectors
            (nod if (p.ndim <= 1 or n.endswith(".bias")) else dec).append(p)
        return dec, nod
    e_d, e_n = split(m.backbone)
    h_d, h_n = split(m.decode_head)
    wd = cfg.WEIGHT_DECAY
    nwd = 0.0 if cfg.NO_DECAY_ON_NORM_BIAS else wd
    return [
        {"params": e_d, "lr": cfg.LR_ENCODER, "weight_decay": wd,  "name": "enc_decay"},
        {"params": e_n, "lr": cfg.LR_ENCODER, "weight_decay": nwd, "name": "enc_nodecay"},
        {"params": h_d, "lr": cfg.LR_DECODER, "weight_decay": wd,  "name": "head_decay"},
        {"params": h_n, "lr": cfg.LR_DECODER, "weight_decay": nwd, "name": "head_nodecay"},
    ]

optimizer = torch.optim.AdamW(param_groups(model), weight_decay=cfg.WEIGHT_DECAY)
LR_IDX_ENC, LR_IDX_DEC = 0, 2

total_steps = cfg.COS_HORIZON_STEPS          # steps, NOT epochs -- this is the arm-matching line

def lr_lambda(step):
    if step < cfg.WARMUP_STEPS:
        return (step + 1) / cfg.WARMUP_STEPS
    prog = (step - cfg.WARMUP_STEPS) / max(1, total_steps - cfg.WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)

class ModelEMA:
    def __init__(self, m, decay):
        self.ema = copy.deepcopy(m).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
        self.decay = decay
    @torch.no_grad()
    def update(self, m, step):
        d = min(self.decay, (1.0 + step) / (10.0 + step))
        msd = m.state_dict()
        for k, v in self.ema.state_dict().items():
            # float buffers here include the BatchNorm running stats, which SHOULD be averaged
            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else: v.copy_(msd[k])

ema = ModelEMA(model, cfg.EMA_DECAY) if cfg.USE_EMA else None
eval_model = ema.ema if cfg.USE_EMA else model

n_par = sum(p.numel() for p in model.parameters())
model.eval()
with torch.no_grad():
    _f = model.backbone(torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda"))
    _head_stride = cfg.IMG_SIZE // _f[ARCH["in_index"][0]].shape[-1]
    probe = forward_logits(model, torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda"))
model.train()
assert probe.shape[-2:] == (cfg.IMG_SIZE, cfg.IMG_SIZE), probe.shape
print(f"params: {n_par/1e6:.2f} M   (expected 4.23 M for MSCAN-T + LightHam @ 6 classes)")
print(f"stage shapes: {[tuple(t.shape[1:]) for t in _f]}")
print(f"head fuses at stride {_head_stride} -> logits "
      f"{cfg.IMG_SIZE//_head_stride}x{cfg.IMG_SIZE//_head_stride}, upsampled {_head_stride}x to 512")
print(f"logits after upsample: {tuple(probe.shape)}")
print(f"cosine horizon: {total_steps} steps (identical in both arms) | EMA: {cfg.USE_EMA}")
print(f"arch_sig {ARCH_SIG} -- must match across arms and across resumes")

## 14. Metrics module

Unchanged. `ConfMat` additionally gets used per base tile in section 20, which is what the
bootstrap resamples.

In [ ]:
from skimage.morphology import skeletonize

class ConfMat:
    def __init__(self, n): self.n = n; self.m = torch.zeros(n, n, dtype=torch.int64, device="cuda")
    def update(self, pred, tgt):
        k = tgt != cfg.IGNORE_INDEX
        if k.sum() == 0: return
        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),
                                 minlength=self.n ** 2).reshape(self.n, self.n)
    def numpy(self): return self.m.cpu().numpy().astype(np.float64)

def metrics_from_cm(cm, names):
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()
    iou = tp / np.maximum(tp + fp + fn, 1e-12)
    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)
    prod = tp / np.maximum(cm.sum(1), 1e-12)
    user = tp / np.maximum(cm.sum(0), 1e-12)
    freq = cm.sum(1) / max(tot, 1e-12)
    po = tp.sum() / max(tot, 1e-12)
    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)
    return {
        "mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)),
        "OA": float(po), "FWIoU": float((freq * iou).sum()),
        "kappa": float((po - pe) / max(1 - pe, 1e-12)),
        "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),
                          "producer_acc": float(prod[i]), "user_acc": float(user[i]),
                          "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)},
        "confusion_matrix_raw": cm.astype(np.int64).tolist(),
        "confusion_matrix_norm": (cm / np.maximum(cm.sum(1, keepdims=True), 1e-12)).tolist(),
    }

def cldice_pair(pred_bin, gt_bin):
    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None
    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))
    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)
    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)
    if tprec + tsens == 0: return 0.0
    return float(2 * tprec * tsens / (tprec + tsens))

## 15. Efficiency metrics

Both arms run the same architecture, so these are identical by construction — they are recorded so
the Stage-4 table can stand alone rather than pointing back at part 1, and because determinism is
now off, these FPS numbers are finally comparable across models.

Two caveats carried over from part 1: `ptflops` hooks `nn.Module`s and therefore misses the NMF's
functional `bmm` calls entirely (the same class of undercount as B0's functional attention matmuls),
and `GMACs_512` is the figure published tables quote while `GFLOPs_512` is 2x that. FPS is
mildly stochastic here because the NMF bases are redrawn per forward pass; do not report more than
one decimal.

In [ ]:
class _FlopsWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return forward_logits(self.m, x)

def efficiency_report(m):
    from ptflops import get_model_complexity_info
    m.eval()
    w = _FlopsWrapper(m)
    with torch.cuda.amp.autocast(enabled=False):
        macs, params = get_model_complexity_info(
            w, (3, cfg.IMG_SIZE, cfg.IMG_SIZE),
            as_strings=False, print_per_layer_stat=False, verbose=False)
    x = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda")
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
        for _ in range(20): forward_logits(m, x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(50): forward_logits(m, x)
        torch.cuda.synchronize()
    return {"params_M": params / 1e6,
            "GMACs_512": macs / 1e9,                # published tables quote this figure
            "GFLOPs_512": macs * 2 / 1e9,           # ptflops returns MACs; FLOPs = 2 x MACs
            "inference_fps_bs1": 50 / (time.time() - t0)}

## 16. Hugging Face persistence and resume

One repo per arm, so `R0` and `R1` never overwrite each other. `last.pt` carries the full
optimizer / scheduler / scaler / EMA state plus the step counter, so a killed session resumes
mid-run rather than restarting.

`segnext_model.py` is pushed **before training starts** and the repo is then re-queried to confirm
it landed. B0 could rebuild itself from `transformers`; this model cannot, so without that file
`best.pt` is unusable outside the kernel that trained it — and Stage 4 produces two checkpoints that
somebody will want to load side by side in six months.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)

def hf_push(local_path, name, msg):
    for attempt in range(3):
        try:
            api.upload_file(path_or_fileobj=local_path, path_in_repo=name,
                            repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)
            return True
        except Exception as e:
            print(f"    push {name} failed ({type(e).__name__}), retry {attempt+1}/3")
            time.sleep(5)
    return False

def hf_try_download(name, repo=None):
    try: return hf_hub_download(repo or HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

# the architecture definition must live alongside the weights -- see the note above
assert Path(MODEL_SRC_PATH).exists(), f"{MODEL_SRC_PATH} missing -- rerun section 13"
hf_push(MODEL_SRC_PATH, "segnext_model.py", f"architecture source (arch_sig {ARCH_SIG})")
assert "segnext_model.py" in {f.rfilename for f in api.repo_info(HF_REPO, token=HF_TOKEN).siblings}, \
    "segnext_model.py did not land in the repo -- do not start training until it does"
print("architecture source verified in", HF_REPO)

def push_model_card(best_metrics, step):
    pc = best_metrics.get("per_class", {})
    L = ["---", "tags: [semantic-segmentation, remote-sensing, segnext, mscan, geonusaf, synthetic-data]",
         "license: apache-2.0", "---", "",
         f"# GeoNUSAF Stage 4 - SegNeXt-T - arm {cfg.ARM} - {cfg.SPLIT_MODE} fold {cfg.FOLD}", "",
         "Kathmandu Valley land-use segmentation, 6 classes, ignore_index=255.", "",
         "| field | value |", "|---|---|",
         "| architecture | MSCAN-T encoder + LightHamHead decoder (SegNeXt, NeurIPS 2022) |",
         f"| arch_sig | `{ARCH_SIG}` |",
         f"| encoder init | ImageNet-1K, {PRETRAIN_COVERAGE*100:.1f}% of tensors loaded |",
         f"| params | {n_par/1e6:.2f} M |",
         f"| head fuse stride | {_head_stride} (stages {ARCH['in_index']}) |",
         f"| NMF rank / steps | R={cfg.MD_R}, {cfg.MD_TRAIN_STEPS} train / {cfg.MD_EVAL_STEPS} eval, rand_init={cfg.MD_RAND_INIT} |",
         f"| arm | {cfg.ARM} ({'real only' if not USE_FAKE else 'real + 804 synthetic'}) |",
         f"| train pairs | {len(tr_recs)} ({len(real_tr_recs)} real + {len(fake_tr_recs)} synthetic) |",
         "| synthetic source | sugam24/geonusaf-stage3-fakepairs-block-fold1, fake_pairs_R1 |",
         f"| split | {cfg.SPLIT_MODE} fold {cfg.FOLD}, train_sha1 `{split_sha1[:12]}` |",
         f"| schedule | {cfg.MAX_STEPS} steps, warmup {cfg.WARMUP_STEPS}, cosine {cfg.COS_HORIZON_STEPS} |",
         f"| lr (head/encoder) | {cfg.LR_DECODER} / {cfg.LR_ENCODER} |",
         f"| class weights | from {cfg.WEIGHTS_FROM} |",
         f"| seed | {cfg.SEED} |",
         f"| best step | {step} |",
         f"| val mIoU | {best_metrics.get('mIoU', float('nan')):.4f} |",
         f"| val mF1 | {best_metrics.get('mF1', float('nan')):.4f} |",
         f"| val OA | {best_metrics.get('OA', float('nan')):.4f} |",
         f"| val kappa | {best_metrics.get('kappa', float('nan')):.4f} |", "",
         "## Per-class (validation, 136 real fold-1 tiles)", "",
         "| class | IoU | F1 |", "|---|---|---|"]
    for k, v in pc.items():
        L.append(f"| {k} | {v['IoU']:.4f} | {v['F1']:.4f} |")
    L += ["", "Validation contains **no synthetic pixels** in either arm.",
          "", "`best.pt` holds `model_state` (EMA weights when EMA is on), `arch`, `arch_sig`, the "
          "run `cfg` and `metrics`. Rebuild with `segnext_model.py` from this same repo. Model code "
          "derives from Visual-Attention-Network/SegNeXt (Apache-2.0).",
          "", "Not comparable to the part-1 fold-1 SegNeXt run: that one had frozen augmentation "
          "(the persistent_workers bug), determinism on, `MIN_DELTA=0.0`, and an epoch-based "
          "schedule."]
    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write("\n".join(L))
    hf_push(p, "README.md", f"card step {step}")

print("HF ready:", HF_REPO)

## 17. Weights & Biases

In [ ]:
import wandb
try:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB = True
except Exception as e:
    print("wandb disabled:", e); WANDB = False

if WANDB:
    wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,
               resume="allow", config={**asdict(cfg), "arch": ARCH, "arch_sig": ARCH_SIG,
                                       "params_M": n_par / 1e6, "n_train_pairs": len(tr_recs),
                                       "n_real": len(real_tr_recs), "n_fake": len(fake_tr_recs)})

## 18. Train / validate loops

`batch_stream()` is an endless iterator over the train loader. It rebuilds the loader's iterator
whenever it runs dry and counts passes for logging, which is the only bookkeeping the step-based
loop needs — augmentation no longer depends on knowing which pass it is on.

Two mechanics worth restating because they are load-bearing: `scaler.unscale_` runs **before**
`clip_grad_norm_`, or the 1.0 threshold would be clipping loss-scaled gradients and mean nothing;
and the EMA updates only on steps the optimizer actually took, so a step the GradScaler skipped on
an inf/NaN check never drags the average toward unused weights.

Validation is the full 408 real val images every time. At 200-step intervals that is 30 validations
over a full run, which costs a few minutes total and buys a properly resolved early-stopping curve.

Watch the first few hundred steps: the head LR is 6e-4 and the Hamburger's output feeds a residual
add. If grad norms sit pinned at the 1.0 clip for the whole warmup rather than just the first ~100
steps, that is the signal the head LR is too hot — and if you drop it, drop it in **both** arms and
restart both.

In [ ]:
PASS_COUNT = {"n": 0}

def batch_stream():
    while True:
        for b in train_ld:
            yield b
        PASS_COUNT["n"] += 1

@torch.no_grad()
def validate(m=None):
    m = m if m is not None else eval_model
    m.eval(); cm = ConfMat(cfg.N_CLASSES); tot, n = 0.0, 0
    for x, y in val_ld:
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(m, x)
        loss, _, _ = criterion(logits, y)
        cm.update(logits.float().argmax(1), y)
        tot += float(loss) * x.size(0); n += x.size(0)
    return tot / max(n, 1), metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)

def train_steps(stream, n_steps, step0):
    model.train()
    tot, seen, gn_acc = 0.0, 0, []
    for k in range(n_steps):
        x, y = next(stream)
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(model, x)
        loss, ce, lv = criterion(logits, y)
        scaler.scale(loss / cfg.GRAD_ACCUM).backward()

        if (k + 1) % cfg.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)
            gn_acc.append(float(gn))
            prev_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            if ema is not None and scaler.get_scale() >= prev_scale:
                ema.update(model, scheduler.last_epoch)

        tot += float(loss) * x.size(0); seen += x.size(0)
        if k % 50 == 0:
            print(f"  s{step0+k} loss {float(loss):.4f} (ce {float(ce):.4f} lov {float(lv):.4f}) "
                  f"gnorm {gn_acc[-1] if gn_acc else 0:.3f}", flush=True)
    return tot / max(seen, 1), float(np.mean(gn_acc)), float(np.max(gn_acc))

## 19. Main loop with automatic resume

Early stopping is on val mIoU with a patience of 8 validations. `MAX_HOURS` stops the session
cleanly and pushes `last.pt`; re-running the notebook with the same `ARM` picks up from that step.

The resume path checks two things before it loads anything: that the checkpoint belongs to **this**
arm, and that its `arch_sig` matches the architecture just built. Both repos are per-arm so the
first should be impossible, and that is exactly why it is cheap to assert.

Expected cost with determinism off: ~0.45-0.7 s/step, so 6000 steps plus 30 validations lands at
roughly **1.5-2.5 h per arm**. VRAM: stage 1 runs at 128x128 with `mlp_ratio=8`, so the widest
activation is 16x256x128x128 — expect 7-10 GB of 16 GB at batch 16 with AMP. This is the tightest
of the lightweight models but it does fit; do not drop the batch size, because MSCAN uses
BatchNorm throughout and is genuinely batch-size sensitive.

In [ ]:
ck = hf_try_download("last.pt")
step, best_miou, best_step, bad, history = 0, -1.0, -1, 0, []
if ck:
    st = torch_load(ck, "cuda")
    assert st.get("arm", cfg.ARM) == cfg.ARM, (
        f"last.pt belongs to arm {st.get('arm')}, this session is {cfg.ARM} -- wrong repo")
    assert st.get("arch_sig", ARCH_SIG) == ARCH_SIG, (
        f"arch_sig mismatch: checkpoint {st.get('arch_sig')} vs current {ARCH_SIG}. The model "
        f"definition or ARCH changed since that run; resuming would blend two architectures.")
    model.load_state_dict(st["model_state"]); optimizer.load_state_dict(st["optim_state"])
    scheduler.load_state_dict(st["sched_state"]); scaler.load_state_dict(st["scaler_state"])
    if ema is not None and st.get("ema_state") is not None:
        ema.ema.load_state_dict(st["ema_state"])
    step = st["step"]; best_miou = st["best_miou"]; best_step = st["best_step"]
    bad = st["bad"]; history = st["history"]
    print(f"RESUMED at step {step} (best mIoU {best_miou:.4f} @ s{best_step})")
else:
    print("fresh start")

torch.cuda.reset_peak_memory_stats()
t_start, best_metrics, stream = time.time(), {}, batch_stream()
n_val = 0
STOP = None

while step < cfg.MAX_STEPS:
    n = min(cfg.VAL_EVERY_STEPS, cfg.MAX_STEPS - step)
    t0 = time.time()
    tr_loss, gn_mean, gn_max = train_steps(stream, n, step)
    step += n; n_val += 1
    va_loss, m = validate()
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    row = {"step": step, "pass": PASS_COUNT["n"], "train_loss": tr_loss, "val_loss": va_loss,
           "overfit_gap": va_loss - tr_loss, "val_mIoU": m["mIoU"], "val_mF1": m["mF1"],
           "val_OA": m["OA"], "val_kappa": m["kappa"], "grad_norm_mean": gn_mean,
           "grad_norm_max": gn_max, "lr_enc": scheduler.get_last_lr()[LR_IDX_ENC],
           "lr_dec": scheduler.get_last_lr()[LR_IDX_DEC], "peak_vram_gb": peak_gb,
           "block_sec": time.time() - t0,
           **{f"IoU/{k}": v["IoU"] for k, v in m["per_class"].items()}}
    history.append(row)
    if WANDB: wandb.log(row, step=step)
    print(f"[s{step}] train {tr_loss:.4f} | val {va_loss:.4f} | mIoU {m['mIoU']:.4f} | "
          f"River {m['per_class']['River']['IoU']:.4f} Road {m['per_class']['Road']['IoU']:.4f} | "
          f"{row['block_sec']:.0f}s | vram {peak_gb:.1f}GB", flush=True)

    if m["mIoU"] > best_miou:
        best_miou, best_step, bad, best_metrics = m["mIoU"], step, 0, m
        p = f"{cfg.OUT_DIR}/best.pt"
        torch.save({"model_state": eval_model.state_dict(),      # EMA weights when EMA is on
                    "arch": ARCH, "arch_sig": ARCH_SIG, "arm": cfg.ARM,
                    "cfg": asdict(cfg), "metrics": m, "step": step}, p)
        hf_push(p, "best.pt", f"best mIoU {best_miou:.4f} @ s{step}")
        push_model_card(best_metrics, step)
        print(f"  -> new best {best_miou:.4f}, pushed")
    else:
        bad += 1

    elapsed_h = (time.time() - t_start) / 3600
    if bad >= cfg.PATIENCE_VALS: STOP = "early stop (patience)"
    elif elapsed_h > cfg.MAX_HOURS: STOP = "session budget"
    elif step >= cfg.MAX_STEPS: STOP = "max steps"

    if (n_val % cfg.PUSH_LAST_EVERY_VALS == 0) or STOP:
        p = f"{cfg.OUT_DIR}/last.pt"
        torch.save({"model_state": model.state_dict(), "optim_state": optimizer.state_dict(),
                    "sched_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
                    "ema_state": (ema.ema.state_dict() if ema is not None else None),
                    "arch_sig": ARCH_SIG, "arm": cfg.ARM,
                    "step": step, "best_miou": best_miou, "best_step": best_step,
                    "bad": bad, "history": history}, p)
        hf_push(p, "last.pt", f"state s{step}")

    if STOP:
        print(f"STOP: {STOP} at step {step} (best {best_miou:.4f} @ s{best_step})")
        break

wall_h = (time.time() - t_start) / 3600
print(f"session done | best mIoU {best_miou:.4f} @ step {best_step} | {wall_h:.2f} h")
if STOP == "session budget":
    print("RE-RUN THIS NOTEBOOK WITH THE SAME ARM -- it resumes from last.pt")

## 20. Final evaluation, per tile

Three things happen here that part 1 did not do.

**Per-base-tile confusion matrices.** The 136 val tiles each get their own 6x6 matrix, summed over
their three flip variants. Summing all 136 reproduces the pooled result exactly, so nothing is lost
— but keeping them separate is what makes the paired bootstrap in section 22 possible.

**A bootstrap confidence interval.** Resampling the 136 tiles with replacement 2000 times gives the
spread of mIoU attributable to *which tiles happened to land in this fold's validation block*.

**The NMF noise floor.** `EVAL_REPEATS` full validation passes on the *same frozen weights*. The
bases are redrawn every forward pass, so these passes disagree, and the spread they show is the
part of any R1 - R0 difference that is evaluation noise rather than data. Part 1 already flagged
this as a few thousandths and told you to expect the final number to sit slightly off the logged
best-step value; here it is measured instead of assumed, because the Stage-4 headline is a
difference and a few thousandths is the same order as the effect being looked for. Pass 0 is the
one that builds `TILE_CM` and every reported number; the extras only produce the spread.

In [ ]:
_bp = f"{cfg.OUT_DIR}/best.pt"
if not os.path.exists(_bp):
    # session resumed past the end, or the loop never improved locally -- pull the best from the Hub
    _bp = hf_try_download("best.pt")
    assert _bp, "no best.pt locally or on the Hub -- training never completed a validation"
    print("best.pt pulled from the Hub")
_ck = torch_load(_bp, "cuda")
assert _ck.get("arch_sig", ARCH_SIG) == ARCH_SIG, "best.pt arch_sig disagrees with this notebook"
best_model = SegNeXt(_ck["arch"]).cuda()
best_model.load_state_dict(_ck["model_state"])
best_model.eval()

torch.manual_seed(cfg.SEED)                    # the reported pass is at least reproducible
VAL_TILE_IDS = sorted(va_ids)
tile_pos = {b: i for i, b in enumerate(VAL_TILE_IDS)}
TILE_CM = np.zeros((len(VAL_TILE_IDS), cfg.N_CLASSES, cfg.N_CLASSES), dtype=np.int64)
cld = {1: [], 2: []}

with torch.no_grad():
    for j in range(0, len(va_recs), cfg.BATCH_SIZE):
        chunk = va_recs[j:j + cfg.BATCH_SIZE]
        xs, ys = zip(*[val_ds[j + k] for k in range(len(chunk))])
        x = torch.stack(xs).cuda(); y = torch.stack(ys).cuda()
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(best_model, x)
        pred = logits.float().argmax(1)
        for b, rec in enumerate(chunk):
            c = ConfMat(cfg.N_CLASSES); c.update(pred[b], y[b])
            TILE_CM[tile_pos[rec[0]]] += c.numpy().astype(np.int64)
        pn, yn = pred.cpu().numpy(), y.cpu().numpy()
        valid = yn != cfg.IGNORE_INDEX
        for b in range(pn.shape[0]):
            for c_ in cld:
                v = cldice_pair((pn[b] == c_) & valid[b], (yn[b] == c_) & valid[b])
                if v is not None: cld[c_].append(v)

CM_TOTAL = TILE_CM.sum(0).astype(np.float64)
final = metrics_from_cm(CM_TOTAL, cfg.CLASS_NAMES)
final["clDice"] = {cfg.CLASS_NAMES[c]: (float(np.mean(v)) if v else None) for c, v in cld.items()}
final["clDice_n_images"] = {cfg.CLASS_NAMES[c]: len(v) for c, v in cld.items()}

# ---- NMF re-sampling noise floor: same weights, extra passes ----
rep_miou = [final["mIoU"]]
for _r in range(1, max(1, cfg.EVAL_REPEATS)):
    _cm = ConfMat(cfg.N_CLASSES)
    with torch.no_grad():
        for x, y in val_ld:
            x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
            with torch.cuda.amp.autocast(enabled=cfg.AMP):
                _lo = forward_logits(best_model, x)
            _cm.update(_lo.float().argmax(1), y)
    rep_miou.append(metrics_from_cm(_cm.numpy(), cfg.CLASS_NAMES)["mIoU"])
final["nmf_eval_mIoU_repeats"] = [float(v) for v in rep_miou]
final["nmf_eval_mIoU_std"] = float(np.std(rep_miou, ddof=1)) if len(rep_miou) > 1 else 0.0
final["nmf_eval_mIoU_range"] = float(max(rep_miou) - min(rep_miou))

def boot_metrics(tile_cm, n_boot, seed=0):
    rs = np.random.RandomState(seed)
    N = tile_cm.shape[0]
    mious, per_cls = [], []
    for _ in range(n_boot):
        pick = rs.randint(0, N, N)
        cm = tile_cm[pick].sum(0).astype(np.float64)
        tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
        den = tp + fp + fn
        iou = np.where(den > 0, tp / np.maximum(den, 1e-12), np.nan)   # absent class -> excluded
        mious.append(float(np.nanmean(iou))); per_cls.append(iou)
    return np.array(mious), np.array(per_cls)

bm, bpc = boot_metrics(TILE_CM, cfg.BOOTSTRAP_N, cfg.SEED)
final["bootstrap"] = {
    "n": cfg.BOOTSTRAP_N, "unit": "val base tile (136)",
    "mIoU_mean": float(bm.mean()), "mIoU_ci95": [float(np.percentile(bm, 2.5)),
                                                 float(np.percentile(bm, 97.5))],
    "per_class_ci95": {n: [float(np.percentile(bpc[:, i], 2.5)),
                           float(np.percentile(bpc[:, i], 97.5))]
                       for i, n in enumerate(cfg.CLASS_NAMES)}}

final.update(efficiency_report(best_model))
final["peak_train_vram_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)
final["train_wall_clock_h_this_session"] = wall_h
final["best_step"] = best_step
final["logged_best_mIoU"] = best_miou
final["pretrain_coverage"] = PRETRAIN_COVERAGE
final["head_fuse_stride"] = _head_stride
final["arch_sig"] = ARCH_SIG
final["run"] = {"model": "SegNeXt-T", "arm": cfg.ARM, "split_mode": cfg.SPLIT_MODE,
                "fold": cfg.FOLD, "seed": cfg.SEED, "n_train": len(tr_recs),
                "n_real": len(real_tr_recs), "n_fake": len(fake_tr_recs),
                "train_sha1": split_sha1, "weights_from": cfg.WEIGHTS_FROM}

print(f"arm {cfg.ARM} | mIoU {final['mIoU']:.4f} "
      f"[{final['bootstrap']['mIoU_ci95'][0]:.4f}, {final['bootstrap']['mIoU_ci95'][1]:.4f}] "
      f"| mF1 {final['mF1']:.4f} | OA {final['OA']:.4f} | kappa {final['kappa']:.4f}")
print("\nper class:")
for i, (k, v) in enumerate(final["per_class"].items()):
    lo, hi = final["bootstrap"]["per_class_ci95"][k]
    print(f"  {k:<14} IoU {v['IoU']:.4f} [{lo:.4f}, {hi:.4f}]  F1 {v['F1']:.4f}")
print("clDice:", {k: (round(v, 4) if v is not None else None)
                  for k, v in final["clDice"].items()})
_d = final["mIoU"] - best_miou
print(f"\nre-evaluation delta vs logged best step: {_d:+.5f}")
print(f"NMF eval noise over {len(rep_miou)} passes on frozen weights: "
      f"std {final['nmf_eval_mIoU_std']:.5f}, range {final['nmf_eval_mIoU_range']:.5f} "
      f"-- {[round(v, 5) for v in rep_miou]}")

## 21. Persist results

`tile_cm_{arm}.npy` is the file section 22 needs from **both** arms. Everything else mirrors the
part-1 outputs so the two stages' result files stay diffable.

In [ ]:
import csv
jp = f"{cfg.OUT_DIR}/results_{tag}.json"
json.dump(final, open(jp, "w"), indent=2)

np.save(f"{cfg.OUT_DIR}/tile_cm_{cfg.ARM}.npy", TILE_CM)
json.dump({"val_tile_ids": VAL_TILE_IDS, "arm": cfg.ARM, "train_sha1": split_sha1,
           "arch_sig": ARCH_SIG},
          open(f"{cfg.OUT_DIR}/tile_index_{cfg.ARM}.json", "w"), indent=2)

flat = {"model": "SegNeXt-T", "arm": cfg.ARM, "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD,
        "seed": cfg.SEED, "n_train": len(tr_recs), "n_fake": len(fake_tr_recs),
        "mIoU": final["mIoU"], "mIoU_lo": final["bootstrap"]["mIoU_ci95"][0],
        "mIoU_hi": final["bootstrap"]["mIoU_ci95"][1],
        "mF1": final["mF1"], "OA": final["OA"], "FWIoU": final["FWIoU"], "kappa": final["kappa"],
        "params_M": final["params_M"], "GMACs_512": final["GMACs_512"],
        "GFLOPs_512": final["GFLOPs_512"],
        "fps_bs1": final["inference_fps_bs1"], "peak_vram_gb": final["peak_train_vram_gb"],
        "best_step": final["best_step"],
        "nmf_eval_std": final["nmf_eval_mIoU_std"], "nmf_eval_range": final["nmf_eval_mIoU_range"],
        "lr_dec": cfg.LR_DECODER, "lr_enc": cfg.LR_ENCODER, "weight_decay": cfg.WEIGHT_DECAY,
        "label_smoothing": cfg.LABEL_SMOOTHING, "cos_horizon_steps": cfg.COS_HORIZON_STEPS,
        "ema": cfg.USE_EMA, "drop_path": cfg.DROP_PATH_RATE,
        "fuse_stage1": cfg.FUSE_STAGE1, "md_r": cfg.MD_R, "md_rand_init": cfg.MD_RAND_INIT,
        "pretrain_coverage": final["pretrain_coverage"], "arch_sig": ARCH_SIG}
for k, v in final["per_class"].items():
    flat[f"IoU_{k}"] = v["IoU"]; flat[f"F1_{k}"] = v["F1"]
for k, v in final["clDice"].items():
    flat[f"clDice_{k}"] = v

cp = f"{cfg.OUT_DIR}/results_{tag}.csv"
with open(cp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)

hp = f"{cfg.OUT_DIR}/history_{tag}.csv"
with open(hp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0])); w.writeheader(); w.writerows(history)

for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"), (hp, f"history_{tag}.csv"),
             (f"{cfg.OUT_DIR}/tile_cm_{cfg.ARM}.npy", f"tile_cm_{cfg.ARM}.npy"),
             (f"{cfg.OUT_DIR}/tile_index_{cfg.ARM}.json", f"tile_index_{cfg.ARM}.json")]:
    hf_push(p, n, "results")
if WANDB: wandb.finish()
print("results pushed to", HF_REPO)

## 22. R0 vs R1 — the actual Stage-4 result

**Run this only after both arms have finished.** It pulls both result sets from the Hub and computes
the comparison the paper reports.

The headline number is a **paired bootstrap**: each of the 2000 resamples draws the same 136 tile
indices for both arms and takes the difference, so tile-to-tile difficulty cancels out instead of
adding two independent variances together. The reported p-value is the fraction of resamples in
which the difference has the opposite sign to the observed one — a two-sided sign test over the
resample distribution, not a t-test, and it makes no normality assumption.

**The NMF noise floor is printed alongside it, and it is not optional reading for this model.** The
combined evaluation noise is `sqrt(std_R0^2 + std_R1^2)`, since the two passes are independent
draws. A delta smaller than roughly twice that is not distinguishable from the decoder redrawing
its bases, whatever the tile bootstrap says — the bootstrap resamples tiles, not base draws, so it
cannot see this source at all.

One thing neither statistic can tell you: both arms ran a single seed, so nothing here covers
variation across training runs. State that in Limitations rather than letting the intervals imply
more than they measure.

In [ ]:
import pandas as pd

def pull_arm(arm):
    repo = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{arm}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
    t = f"{arm}_{cfg.SPLIT_MODE}_fold{cfg.FOLD}"
    res = json.load(open(hf_hub_download(repo, f"results_{t}.json", token=HF_TOKEN)))
    cm  = np.load(hf_hub_download(repo, f"tile_cm_{arm}.npy", token=HF_TOKEN))
    idx = json.load(open(hf_hub_download(repo, f"tile_index_{arm}.json", token=HF_TOKEN)))
    return res, cm, idx

try:
    R0, CM0, IX0 = pull_arm("R0")
    R1, CM1, IX1 = pull_arm("R1")
except Exception as e:
    print(f"both arms not finished yet ({type(e).__name__}) -- run this after the second one")
    R0 = None

if R0 is not None:
    assert IX0["val_tile_ids"] == IX1["val_tile_ids"], "arms validated on different tiles"
    assert IX0["train_sha1"] == IX1["train_sha1"], "arms used different real splits"
    assert IX0.get("arch_sig") == IX1.get("arch_sig"), "arms used different architectures"
    N = CM0.shape[0]

    def iou_from(cm):
        tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
        den = tp + fp + fn
        return np.where(den > 0, tp / np.maximum(den, 1e-12), np.nan)

    rs = np.random.RandomState(cfg.SEED)
    d_miou, d_cls = [], []
    for _ in range(cfg.BOOTSTRAP_N):
        pick = rs.randint(0, N, N)                       # SAME tiles for both arms
        i0 = iou_from(CM0[pick].sum(0).astype(np.float64))
        i1 = iou_from(CM1[pick].sum(0).astype(np.float64))
        d_miou.append(np.nanmean(i1) - np.nanmean(i0)); d_cls.append(i1 - i0)
    # per-class deltas can carry nan if a class vanished from a resample; nan-aware percentiles below
    d_miou = np.array(d_miou); d_cls = np.array(d_cls)

    obs = R1["mIoU"] - R0["mIoU"]
    p = min(1.0, float(np.mean(d_miou <= 0) if obs > 0 else np.mean(d_miou >= 0)) * 2)
    print(f"{'metric':<16}{'R0':>10}{'R1':>10}{'delta':>10}{'95% CI on delta':>24}")
    print(f"{'mIoU':<16}{R0['mIoU']:10.4f}{R1['mIoU']:10.4f}{obs:+10.4f}"
          f"   [{np.percentile(d_miou,2.5):+.4f}, {np.percentile(d_miou,97.5):+.4f}]")
    for k in ["mF1", "OA", "FWIoU", "kappa"]:
        print(f"{k:<16}{R0[k]:10.4f}{R1[k]:10.4f}{R1[k]-R0[k]:+10.4f}")
    print(f"\npaired-bootstrap two-sided p on mIoU: {p:.4f}  (n={cfg.BOOTSTRAP_N} resamples)")

    s0 = R0.get("nmf_eval_mIoU_std", 0.0); s1 = R1.get("nmf_eval_mIoU_std", 0.0)
    nmf = float(np.hypot(s0, s1))
    print(f"NMF evaluation noise: R0 std {s0:.5f} | R1 std {s1:.5f} | combined {nmf:.5f}")
    if nmf > 0:
        print(f"  |delta| / combined noise = {abs(obs)/nmf:.1f}x  "
              f"({'above' if abs(obs) > 2*nmf else 'NOT above'} 2x -- "
              f"{'the delta survives the decoder resampling' if abs(obs) > 2*nmf else 'report it as within evaluation noise'})")

    print(f"\n{'class':<14}{'IoU R0':>9}{'IoU R1':>9}{'delta':>9}{'95% CI on delta':>24}")
    rows = []
    for i, nm in enumerate(cfg.CLASS_NAMES):
        a, b = R0["per_class"][nm]["IoU"], R1["per_class"][nm]["IoU"]
        lo, hi = np.nanpercentile(d_cls[:, i], 2.5), np.nanpercentile(d_cls[:, i], 97.5)
        star = " *" if (lo > 0 or hi < 0) else ""
        print(f"{nm:<14}{a:9.4f}{b:9.4f}{b-a:+9.4f}   [{lo:+.4f}, {hi:+.4f}]{star}")
        rows.append({"class": nm, "IoU_R0": a, "IoU_R1": b, "delta": b - a,
                     "ci_lo": lo, "ci_hi": hi})
    print("\n* = 95% CI on the difference excludes zero")

    print(f"\n{'clDice':<14}{'R0':>9}{'R1':>9}{'delta':>9}")
    for k in R0["clDice"]:
        a, b = R0["clDice"][k], R1["clDice"][k]
        if a is not None and b is not None:
            print(f"{k:<14}{a:9.4f}{b:9.4f}{b-a:+9.4f}")

    comp = {"model": "SegNeXt-T",
            "mIoU_R0": R0["mIoU"], "mIoU_R1": R1["mIoU"], "delta_mIoU": obs,
            "delta_mIoU_ci95": [float(np.percentile(d_miou, 2.5)),
                                float(np.percentile(d_miou, 97.5))],
            "p_two_sided": p, "n_boot": cfg.BOOTSTRAP_N, "per_class": rows,
            "nmf_eval_std": {"R0": s0, "R1": s1, "combined": nmf},
            "best_step": {"R0": R0["best_step"], "R1": R1["best_step"]}}
    json.dump(comp, open(f"{cfg.OUT_DIR}/stage4_R0_vs_R1.json", "w"), indent=2, default=float)
    pd.DataFrame(rows).to_csv(f"{cfg.OUT_DIR}/stage4_R0_vs_R1.csv", index=False)
    hf_push(f"{cfg.OUT_DIR}/stage4_R0_vs_R1.json", "stage4_R0_vs_R1.json", "arm comparison")
    hf_push(f"{cfg.OUT_DIR}/stage4_R0_vs_R1.csv", "stage4_R0_vs_R1.csv", "arm comparison")

## 23. Standalone inference from Hugging Face

Fully self-contained: point `INFER_REPO` at either arm's repo and it downloads **both** the weights
and `segnext_model.py`, then rebuilds the network from the `arch` dict embedded in `best.pt`.
Nothing above this cell is required — paste it into a fresh notebook as-is.

This is the one place SegNeXt needs more machinery than B0, and it is why section 16 pushes the
source before training rather than after.

In [ ]:
def load_from_hf(repo_id, token=None, device="cuda"):
    import torch, numpy as np, cv2, types
    import torch.nn.functional as F
    from huggingface_hub import hf_hub_download
    # the architecture travels with the weights -- see section 16
    try:
        src = open(hf_hub_download(repo_id, "segnext_model.py", token=token)).read()
        mod = types.ModuleType("segnext_model")
        exec(compile(src, "segnext_model.py", "exec"), mod.__dict__)
    except Exception as e:
        # repo predates the architecture-source push; fall back to the class in this
        # kernel. Only valid because section 13 above defines the identical model.
        if "SegNeXt" not in globals():
            raise RuntimeError(
                f"{repo_id} has no segnext_model.py and no SegNeXt class is defined here. "
                f"Upload segnext_model.py to the repo, then retry.") from e
        print(f"  NOTE: {repo_id} has no segnext_model.py -- using the in-kernel class.")
        mod = types.SimpleNamespace(SegNeXt=globals()["SegNeXt"])
    _p = hf_hub_download(repo_id, "best.pt", token=token)
    try:                                              # torch>=2.6 weights_only default
        ck = torch.load(_p, map_location=device, weights_only=True)
    except Exception:
        ck = torch.load(_p, map_location=device, weights_only=False)
    m = mod.SegNeXt(ck["arch"]).to(device).eval()
    m.load_state_dict(ck["model_state"])
    return m, ck["cfg"], ck["metrics"]

@torch.no_grad()
def predict_paths(model, c, paths, device="cuda", batch=8):
    import torch, numpy as np, cv2
    import torch.nn.functional as F
    mean = np.array([0.485, 0.456, 0.406], np.float32)
    std  = np.array([0.229, 0.224, 0.225], np.float32)
    out = []
    for i in range(0, len(paths), batch):
        xs = []
        for p in paths[i:i + batch]:
            im = cv2.imread(str(p), cv2.IMREAD_COLOR)
            im = cv2.resize(im, (c["IMG_SIZE"], c["IMG_SIZE"]), interpolation=cv2.INTER_AREA)
            a = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            xs.append(torch.from_numpy(((a - mean) / std).transpose(2, 0, 1)))
        x = torch.stack(xs).to(device)
        with torch.cuda.amp.autocast():
            lo = model(x)
            lo = F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)
            out.append(lo.float().argmax(1).cpu().numpy())
    return np.concatenate(out)

INFER_REPO = HF_REPO
inf_model, inf_cfg, inf_metrics = load_from_hf(INFER_REPO, token=HF_TOKEN)
print(f"loaded {INFER_REPO} | val mIoU {inf_metrics['mIoU']:.4f}")
demo = [IMG_DIR / r[2] for r in va_recs[:4]]
preds = predict_paths(inf_model, inf_cfg, demo)
print("pred shape:", preds.shape, "| classes present:", np.unique(preds).tolist())

## 24. Qualitative triptychs and the confusion matrix

Same figures as part 1. When both arms are done, the useful comparison is the *same* val tiles under
R0 and R1 side by side — the tiles selected here are this arm's worst cases, so run the section in
both sessions and diff the pairs by filename.

What to look for specifically in this model: SegNeXt's logits come out at 64x64 and are upsampled
8x, so its road edges quantise to 8 px steps. If R1's roads are *less* fragmented but no less
blocky, the synthetic data is helping continuity and not resolution, which is exactly the claim the
`FUSE_STAGE1` ablation would test next.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]
PALETTE_RGB[255] = (0, 0, 0)
def colorize(lab): return PALETTE_RGB[lab]

per_tile_miou = []
for i, b in enumerate(VAL_TILE_IDS):
    cm = TILE_CM[i].astype(np.float64)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    per_tile_miou.append((float(np.nanmean(tp / np.maximum(tp + fp + fn, 1e-12))), b))
per_tile_miou.sort(key=lambda t: t[0])
worst, best = per_tile_miou[:3], per_tile_miou[-3:][::-1]

def triptych(items, title):
    fig, axes = plt.subplots(len(items), 3, figsize=(13, 4.3 * len(items)))
    axes = np.atleast_2d(axes)
    for r, (score, base_id) in enumerate(items):
        rec = groups[base_id][0]
        pos = va_recs.index(rec)
        _, _, iname, mname = rec
        img = cv2.cvtColor(cv2.imread(str(CACHE / "images" / iname)), cv2.COLOR_BGR2RGB)
        gt = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        x, _ = val_ds[pos]
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
            pr = forward_logits(best_model, x[None].cuda()).float().argmax(1)[0].cpu().numpy()
        pr_vis = np.where(gt == 255, 255, pr).astype(np.uint8)
        for c, (im, t) in enumerate([(img, f"tile {base_id}"), (colorize(gt), "ground truth"),
                                     (colorize(pr_vis), f"{cfg.ARM} prediction (mIoU {score:.3f})")]):
            axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")
    handles = [Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)]
    handles.append(Patch(color="black", label="ignore/unannotated"))
    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8, frameon=False)
    fig.suptitle(f"{title} - SegNeXt-T {cfg.ARM}, {cfg.SPLIT_MODE} fold {cfg.FOLD}", fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    p = f"{cfg.OUT_DIR}/qual_{title.replace(' ','_')}_{tag}.png"
    plt.savefig(p, dpi=130, bbox_inches="tight"); plt.show()
    hf_push(p, os.path.basename(p), "qualitative")

triptych(best, "best cases")
triptych(worst, "worst cases")

cmn = np.array(final["confusion_matrix_norm"])
fig, ax = plt.subplots(figsize=(7.2, 6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(cfg.CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Row-normalized confusion - SegNeXt-T {cfg.ARM}, {cfg.SPLIT_MODE} fold {cfg.FOLD}")
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cmn[i, j] > 0.5 else "black")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
p = f"{cfg.OUT_DIR}/confusion_{tag}.png"
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
hf_push(p, os.path.basename(p), "confusion matrix")

## 25. Run checklist

1. Internet **ON**. Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`. Internet also fetches the MSCAN-T
   ImageNet weights; without it, attach `mscan_t_20230227-119e8c9f.pth` as a Kaggle dataset and
   section 13 finds it by rglob.
2. Attach **both** inputs: `sugam24/satellite-datasets` and
   `sugam24/geonusaf-fake-pairs-block-fold1`. Point `PREBUILT_CACHE` at your 512 cache if you have
   one saved — section 6 becomes a no-op and you save ~15 min in each of two sessions.
3. **Session 1:** `ARM = "R0"`. Run top to bottom through section 21, then 23-24.
   Section 22 will print "both arms not finished yet" — expected, skip it.
4. **Session 2:** `ARM = "R1"`. Same run. Section 22 now produces the comparison.
5. If a session hits the 11 h budget, re-run with the same `ARM`; it resumes from `last.pt`.

### Asserts that must pass before you walk away
- Section 5: `channel-order check PASSED`
- Section 8: `four-way agreement with S1/S2/S3 ASSERTED` — if this fails, the real split disagrees
  with what the generators were trained on and the whole experiment is void
- Section 9 (R1 only): 804 synthetic pairs loaded, no synthetic record in validation
- Section 11: `augmentation liveness check PASSED`
- Section 13: `params: 4.23 M`, `pretrained: loaded N/N backbone tensors (100.0%)`,
  `head fuses at stride 8 -> logits 64x64`, `logits after upsample: (1, 6, 512, 512)`
- Section 16: `architecture source verified in <repo>` — this re-queries the repo after the upload,
  so it is proof the file is there. Without it `best.pt` cannot be rebuilt outside this kernel.
- Both sessions must print the **same `arch_sig`**. If they do not, the arms are not comparable.

### If it OOMs
Do **not** reach for `GRAD_ACCUM` — the step budget counts loop iterations, so accumulation would
halve the effective cosine horizon and break the "6000 steps == part 1's 120 epochs" identity that
makes R0 comparable to anything. `PRELOAD_RAM=False` frees ~2.2 GB of host RAM (not VRAM); for VRAM,
the honest options are a smaller `IMG_SIZE` (which changes the protocol for every model) or
accepting the crash and reporting it. At batch 16 with AMP, part 1 measured 7-10 GB of 16 GB, so it
should not come to this.

### Reading the result
Section 22's mIoU delta is the Stage-4 headline, and the per-class table is where the actual
argument lives. River and Road are the classes the synthetic data was supposed to help; a `*` next
to them means the 95% tile-bootstrap interval on the difference excludes zero. **Check the delta
against the NMF noise floor printed just above it** — this is the one model in the lineup whose
evaluation is stochastic on frozen weights, and a delta inside that floor is not a result.

**A negative delta is a publishable result.** "Synthetic pairs at 50% mixing did not improve fold-1
segmentation, and here is the layout-fidelity and memorization evidence for why" is a real
contribution given Stage 3's numbers are already measured. Do not tune the mixing ratio against the
validation set to find a positive number — that is selection on the test set, and with one fold and
one seed it would be indefensible.

### What this run cannot support
One seed, one fold, one architecture, one mixing ratio. Every claim has to be scoped to those four
facts in the paper. The tile bootstrap covers tile sampling, the NMF repeats cover decoder
resampling, and **neither covers run-to-run variation**, which is unmeasured here.

### For the paper
SegNeXt is Apache-2.0 (Visual-Attention-Network, Tsinghua). Cite Guo et al., *SegNeXt*, NeurIPS
2022, and Geng et al., *Is Attention Better Than Matrix Decomposition?*, ICLR 2021, for the
Hamburger decoder. Add SegNeXt to the protocol-deviation ledger row for the frozen-augmentation bug
— it had the same defect as U-Net, SegFormer-B0 and TC-SegFormer — and note that its part-1
`MIN_DELTA=0.0` is a second, independent reason the published fold-1 number is not reusable as an
R0 baseline.